# Temperature

In [ ]:
!python "/content/MLME26/eval/Temperature_eomt.py" \
    --input "/content/drive/MyDrive/utils_MLME26/datasets/RoadObsticle21/images/*.*" \
    --method msp

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import h5py
import os
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score


# CONFIGURAZIONE
FILE_PATH = "val_data.h5"          # file creato con lo script Temperature_eomt.py che deve contenere i logits delle classi
T_VALUES = [0.5, 0.75, 1.0, 1.1, 2]   # temperature da testare
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 100000                # blocchi per non saturare la RAM

def run_temperature_analysis():
    if not os.path.exists(FILE_PATH):
        print(f"Errore: Il file {FILE_PATH} non esiste.")
        return

    with h5py.File(FILE_PATH, 'r') as hf:
        if 'logits' not in hf:
            print("Errore: nel file HDF5 manca il dataset 'logits'")
            return
        logits_dset = hf['logits']
        labels_dset = hf['labels']

        # Verifico che i logits abbiano 19 canali
        if logits_dset.shape[1] != 19:
            print(f"Errore: i logits hanno {logits_dset.shape[1]} canali, ma devono essere 19 (classi semantiche).")
            print("Assicurati che il file HDF5 sia stato generato con logits completi (19 canali).")
            return

        num_total_pixels = logits_dset.shape[0]
        print(f"--- Analisi Stream-based su {num_total_pixels} pixel ---")
        print(f"Device: {DEVICE}")

        auprc_list, fpr95_list, objective_list = [], [], []

        with torch.no_grad():
            for t in T_VALUES:
                all_scores = []
                all_labels = []

                # Leggo il file a blocchi pe ron saturare la RAM
                for i in range(0, num_total_pixels, BATCH_SIZE):
                    end_idx = min(i + BATCH_SIZE, num_total_pixels)

                    batch_logits_cpu = logits_dset[i:end_idx]   # (B, 19)
                    batch_labels_cpu = labels_dset[i:end_idx]

                    # Filtro solo pixel con label 0 (ID) o 1 (OOD)
                    valid_mask = (batch_labels_cpu == 0) | (batch_labels_cpu == 1)
                    if not np.any(valid_mask):
                        continue

                    # Sposto su GPU solo i pixel validi
                    batch_logits_gpu = torch.from_numpy(batch_logits_cpu[valid_mask]).float().to(DEVICE)
                    batch_labels_valid = batch_labels_cpu[valid_mask]

                    # Calcolo MSP in modo identico a evalAnomaly_eomt.py
                    # Applico la temperatura ai logit (divisione per T)
                    scaled_logits = batch_logits_gpu / t
                    # Softmax + max_prob
                    probs = torch.softmax(scaled_logits, dim=1)
                    max_prob, _ = torch.max(probs, dim=1)
                    # Anomaly score = 1 - max_prob
                    scores = 1.0 - max_prob

                    all_scores.append(scores.cpu().numpy())
                    all_labels.append(batch_labels_valid)

                    # Pulizia GPU
                    del batch_logits_gpu, scaled_logits, probs, max_prob, scores
                    if DEVICE.type == 'cuda':
                        torch.cuda.empty_cache()

                #Unisco i risultati di tutti i blocchi per questa temperatura
                full_scores = np.concatenate(all_scores)
                full_labels = np.concatenate(all_labels)

                #Calcolo le metriche
                auprc = average_precision_score(full_labels, full_scores)
                fpr95 = fpr_at_95_tpr(full_scores, full_labels)
                objective = (auprc - fpr95) / 2

                auprc_list.append(auprc)
                fpr95_list.append(fpr95)
                objective_list.append(objective)

                print(f"Temp: {t:5.1f} | AuPRC: {auprc:.4f} | FPR95: {fpr95:.4f} | Obj: {objective:.4f}")

    
    # Grafico finale
    best_idx = np.argmax(objective_list)
    best_t = T_VALUES[best_idx]

    plt.figure(figsize=(10, 6))
    plt.plot(T_VALUES, auprc_list, label='AuPRC', color='blue', marker='o')
    plt.plot(T_VALUES, fpr95_list, label='FPR95', color='red', marker='s')
    plt.plot(T_VALUES, objective_list, label='Objective', color='green', linestyle='--', marker='d')
    plt.axvline(x=best_t, color='black', linestyle=':', alpha=0.7)
    plt.title(f'Ottimizzazione Temperatura (Miglior T = {best_t:.2f})', fontsize=14)
    plt.xlabel('Temperatura', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_temperature_analysis()